# 00b · Lift warm-up: closed-loop evaluation analysis

**Purpose.** Turn the paired rollout results of warm-up 0.4 into two figures and check the
statistics behind docs/BASELINE.md: how success grows with training, where it plateaus, and
why a *paired* evaluation can separate checkpoints whose confidence intervals overlap.
The same protocol (paired seeds, Wilson intervals, McNemar, separate selection and report
seeds) is reused for every "is A better than B" question in phases 2 and 3.

| | |
|---|---|
| Track | A (upstream policy and rollouts; analysis only, no simulation) |
| Kernel | sim venv (`$WORK/franka-il/venvs/sim`); needs `numpy`, `scipy`, `matplotlib` |
| Inputs | `results/lift_eval_episodes.csv` (committed; one row per episode, built from the eval jobs' `eval.json`) |
| Outputs | `docs/figures/lift_eval_success.png`, `docs/figures/lift_eval_paired.png` |
| Runtime | seconds, login node |

Prerequisite: none; the CSV is in the repo. §0b rebuilds it from the eval jobs on CLAIX
and is off by default.

**Success checks** (each section ends with an `assert`):
1. Seed 0: a 10 × 50 success matrix (epochs 20–200 × episodes 0–49); successes per epoch match BASELINE.md.
2. The Wilson interval matches `scipy` and the BASELINE.md table.
3. Epoch 140 vs 160: 0 episodes succeed only at 140, 6 only at 160; exact McNemar p = 0.031.

## 0. Setup and data

In [16]:
import csv
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CSV = REPO / "results" / "lift_eval_episodes.csv"
FIG_DIR = REPO / "docs" / "figures"

### 0b. (CLAIX only) Rebuild the CSV from the eval jobs

Off by default. Each job's `eval.json` holds per-episode records (`success`, `steps`,
`cube_init_pos`). The failure-video re-runs (5863826/5863827) duplicate existing rows and
are left out.

In [17]:
REBUILD = False
EVAL_JOBS = {0: [5844550, 5844568, 5844610, 5844611, 5844612], 1: [5856229]}  # seed -> jobs

if REBUILD:
    import json
    eval_root = Path(os.environ["HPCWORK"]) / "franka-il/runs/lift_eval"
    rows = []
    for seed, jobs in EVAL_JOBS.items():
        for job in jobs:
            report = json.loads((eval_root / str(job) / "eval.json").read_text())
            assert report["settings"]["seed"] == seed and report["paired"]
            for r in report["results"]:
                for e in r["episodes"]:
                    rows.append({"seed": seed, "epoch": r["epoch"], "episode": e["episode"],
                                 "success": int(e["success"]), "steps": e["steps"],
                                 "cube_x": e["cube_init_pos"][0], "cube_y": e["cube_init_pos"][1],
                                 "eval_job": job})
    rows.sort(key=lambda d: (d["seed"], d["epoch"], d["episode"]))
    CSV.parent.mkdir(exist_ok=True)
    with CSV.open("w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=list(rows[0]))
        w.writeheader()
        w.writerows(rows)
    print(f"wrote {len(rows)} rows")

In [18]:
with CSV.open() as fh:
    rows = [{k: (float(v) if k.startswith("cube") else int(v)) for k, v in r.items()}
            for r in csv.DictReader(fh)]
print(len(rows), "rows; columns:", list(rows[0]))
print(rows[0])

550 rows; columns: ['seed', 'epoch', 'episode', 'success', 'steps', 'cube_x', 'cube_y', 'eval_job']
{'seed': 0, 'epoch': 20, 'episode': 0, 'success': 0, 'steps': 400, 'cube_x': 0.00883, 'cube_y': 0.00692, 'eval_job': 5844550}


## 1. From episode rows to a checkpoint × episode matrix

**Why rollouts, not loss.** Behaviour cloning is trained open loop (predict the demo action
from a demo state) but deployed closed loop: each action changes the next state, and small
errors move the robot into states the demos never show (compounding error). Offline loss
cannot see this. Here it is even uninformative: the logged validation loss stays ≈2×10⁶
because eval-mode GMM std is clamped to 1e-4 (BASELINE.md). Checkpoints are therefore
ranked by closed-loop success.

**Why a matrix.** Paired evaluation means **episode *i* starts from the same initial state
for every checkpoint** (same env seed). Rows = checkpoints, columns = episodes: each column
is one initial state, so reading down a column compares checkpoints on equal terms, with
the difficulty of the start state held fixed.

In [19]:
def to_matrix(rows, seed, key):
    sel = [r for r in rows if r["seed"] == seed]
    epochs = np.array(sorted({r["epoch"] for r in sel}))
    n_ep = max(r["episode"] for r in sel) + 1
    M = np.full((len(epochs), n_ep), -1)        # -1 = missing; checked below
    row_of = {e: i for i, e in enumerate(epochs)}
    for r in sel:                                # placement by index, so row order is irrelevant
        M[row_of[r["epoch"]], r["episode"]] = r[key]
    assert (M >= 0).all(), "missing (epoch, episode) entries"
    return epochs, M


epochs, S0 = to_matrix(rows, seed=0, key="success")
_, T0 = to_matrix(rows, seed=0, key="steps")
S0 = S0.astype(bool)

assert S0.shape == (10, 50) and list(epochs) == list(range(20, 201, 20))
assert S0.sum(axis=1).tolist() == [1, 15, 14, 14, 35, 39, 44, 50, 49, 50]   # BASELINE.md
assert (T0[~S0] == 400).all()   # every failure runs to the horizon
print("successes per epoch:", dict(zip(epochs.tolist(), S0.sum(axis=1).tolist())))

successes per epoch: {20: 1, 40: 15, 60: 14, 80: 14, 100: 35, 120: 39, 140: 44, 160: 50, 180: 49, 200: 50}


## 2. Wilson score interval

**Why an interval at all.** 50 rollouts estimate a success *probability*; 44/50 and 50/50
are samples, not the true rates. Without intervals, "epoch 200 is best" would just be
noise ranking.

**Why Wilson, not Wald.** With *k* successes in *n* episodes, the Wald interval
p̂ ± z·√(p̂(1−p̂)/n) collapses to [100%, 100%] at 50/50 (it claims certainty) and can leave
[0, 1]. Good policies live near 100%, exactly where Wald fails. Wilson instead solves
|p̂ − p| ≤ z·√(p(1−p)/n) for *p* (the standard error uses the unknown *p*, not p̂):

$$
\text{center} = \frac{\hat p + \frac{z^2}{2n}}{1 + \frac{z^2}{n}}, \qquad
\text{half} = \frac{z}{1 + \frac{z^2}{n}} \sqrt{\frac{\hat p(1-\hat p)}{n} + \frac{z^2}{4n^2}}
$$

At 50/50 the lower bound simplifies to n/(n + z²) = 50/53.84 ≈ 92.9%: a perfect score on
50 episodes is still compatible with a 93% policy.

In [20]:
def wilson(k, n, z=1.96):
    p = np.asarray(k, dtype=float) / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    half = z / denom * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))
    return center - half, center + half


k = S0.sum(axis=1)
lo, hi = wilson(k, 50)
for e, kk, l, h in zip(epochs, k, lo, hi):
    ref = stats.binomtest(int(kk), 50).proportion_ci(confidence_level=0.95, method="wilson")
    assert abs(l - ref.low) < 1e-3 and abs(h - ref.high) < 1e-3, (e, l, h, ref)
    print(f"epoch {e:3d}: {kk:2d}/50  95% Wilson {100*l:5.1f}-{100*h:5.1f}%")
assert round(100 * lo[-1]) == 93 and round(100 * lo[4]) == 56 and round(100 * hi[4]) == 81

epoch  20:  1/50  95% Wilson   0.4- 10.5%
epoch  40: 15/50  95% Wilson  19.1- 43.8%
epoch  60: 14/50  95% Wilson  17.5- 41.7%
epoch  80: 14/50  95% Wilson  17.5- 41.7%
epoch 100: 35/50  95% Wilson  56.2- 80.9%
epoch 120: 39/50  95% Wilson  64.8- 87.2%
epoch 140: 44/50  95% Wilson  76.2- 94.4%
epoch 160: 50/50  95% Wilson  92.9-100.0%
epoch 180: 49/50  95% Wilson  89.5- 99.6%
epoch 200: 50/50  95% Wilson  92.9-100.0%


### Figure 1: success rate vs training epoch

What to read from it:
- **Plateau, not argmax.** From epoch 160 the intervals of 160/180/200 overlap almost fully;
  the data cannot rank them, so the shaded region is called a plateau and the last
  checkpoint is taken (tied best) instead of the highest point.
- **Selection vs report.** Seed 0 is used to *choose* the checkpoint, so the chosen score on
  seed 0 is optimistically biased (max over noisy estimates). Seed 1 (fresh initial states,
  never used for choosing) gives the unbiased number, drawn separately in red. Same idea as
  validation vs test split.
- **Lower panel.** Median steps of successful episodes keep falling (45 → 41) after success
  saturates: the policy grasps more decisively, which the success rate alone cannot show.

In [21]:
_, S1 = to_matrix(rows, seed=1, key="success")          # epoch 200 only
k1 = int(S1.sum()); lo1, hi1 = wilson(k1, S1.shape[1])
p0 = k / 50
med_steps = [np.median(T0[i][S0[i]]) for i in range(len(epochs))]

fig, (ax, ax2) = plt.subplots(2, 1, figsize=(6.4, 4.8), sharex=True,
                              gridspec_kw={"height_ratios": [3, 1.2]})
ax.errorbar(epochs, 100 * p0, yerr=[100 * (p0 - lo), 100 * (hi - p0)], fmt="o-", color="C0",
            capsize=3, label="seed 0 (selection), 95% Wilson")
ax.errorbar([206], [100 * k1 / S1.shape[1]], yerr=[[100 * (k1 / 50 - lo1)], [100 * (hi1 - k1 / 50)]],
            fmt="s", color="C3", capsize=3, label=f"seed 1 (report), epoch 200: {k1}/50")
ax.axvspan(150, 210, color="0.92", zorder=0)
ax.text(155, 8, "plateau", color="0.4")
ax.set_ylabel("success rate (%)"); ax.set_ylim(0, 105); ax.legend(loc="center right", fontsize=8)
ax.set_title("BC-RNN, Lift ph, low-dim (state-based): 50 paired rollouts per checkpoint", fontsize=9)
ax2.plot(epochs, med_steps, "o-", color="0.3")
ax2.set_ylabel("median steps\n(successes)"); ax2.set_xlabel("training epoch")
ax2.set_xticks(epochs)
fig.tight_layout()
FIG_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG_DIR / "lift_eval_success.png", dpi=120)
plt.show()

## 3. Paired comparison: epoch 140 vs 160

The intervals of 140 (44/50, 76–94%) and 160 (50/50, 93–100%) overlap, so comparing the two
marginal rates alone is inconclusive. But both faced the *same* 50 initial states. Episodes
where both succeed (or both fail) carry no information about which is better; only the
**discordant** episodes do. Under H₀ "equally good", each discordant episode is a fair coin
flip, so the exact McNemar test is a binomial test on the discordant counts: 6 discordant,
all in favour of 160, has probability 2 × (1/2)⁶ ≈ 0.031.

Pairing removes the between-state variance (easy vs hard cube placements) from the
comparison, which is why it is more sensitive than comparing two independent rates.

In [22]:
def discordant(a, b):
    a, b = np.asarray(a, dtype=bool), np.asarray(b, dtype=bool)
    return np.flatnonzero(a & ~b), np.flatnonzero(~a & b)


row = {e: i for i, e in enumerate(epochs)}
only_140, only_160 = discordant(S0[row[140]], S0[row[160]])
b, c = len(only_140), len(only_160)
p_mcnemar = stats.binomtest(b, b + c, 0.5).pvalue          # exact, two-sided
print(f"only 140: {b} {only_140.tolist()} | only 160: {c} {only_160.tolist()} | p = {p_mcnemar:.3f}")
assert (b, c) == (0, 6) and round(p_mcnemar, 3) == 0.031

only 140: 0 [] | only 160: 6 [2, 11, 24, 25, 35, 47] | p = 0.031


### Figure 2: paired outcomes (seed 0)

One column per initial state. What to read from it:
- **Columns, not rows.** A column that fails for many checkpoints is a hard initial state;
  episode 44 succeeds already at epoch 20 (the only success there), an easy one.
- **The 140 → 160 change** (red): every discordant pair goes the same way, the visual form of
  the McNemar result.
- **Late failures look random, not structural.** Epoch 180 fails episode 5, which epochs 160
  and 200 solve; no column fails systematically after the plateau (see the failure modes
  in BASELINE.md: missed grasp without recovery, divergence).

In [23]:
from matplotlib.colors import ListedColormap

fig, ax = plt.subplots(figsize=(9, 3.2))
ax.imshow(S0, aspect="auto", cmap=ListedColormap(["#d9d9d9", "#2b8cbe"]), interpolation="nearest")
ax.set_yticks(range(len(epochs)), [f"epoch {e}  ({s}/50)" for e, s in zip(epochs, k)], fontsize=8)
ax.set_xticks(range(0, 50, 5)); ax.set_xlabel("episode (= paired initial state)")
ax.set_xticks(np.arange(-0.5, 50, 1), minor=True); ax.set_yticks(np.arange(-0.5, len(epochs), 1), minor=True)
ax.grid(which="minor", color="white", linewidth=0.6); ax.tick_params(which="minor", length=0)
for j in only_160:   # succeed at 160, fail at 140
    ax.plot([j, j], [row[140], row[160]], color="C3", linewidth=1.5)
ax.set_title("Seed 0 paired outcomes (blue = success). Red: fail at 140, succeed at 160 "
             f"(exact McNemar p = {p_mcnemar:.3f})", fontsize=9)
fig.tight_layout()
fig.savefig(FIG_DIR / "lift_eval_paired.png", dpi=120)
plt.show()

## 4. Open questions

Think these through before phase 2; the conclusions become rules in
docs/EXPERIMENT_PROTOCOL.md (evaluation section).
1. If seed 1 were also used to choose the checkpoint, what would 49/50 still mean, and
   what would you need to report an unbiased number?
   Ans: Seeds decide the randomized part of the rollout environment, especially the inital position and rotation of the object, which seems to be important for our model. If seed 1 were also used to choose checkpoint, then every checkpoints need to run the rollout with seed 1 as well. 
2. How many episodes would be needed to tell a 98% policy from a 100% one (or, more
   realistically, 90% from 98%) with a paired test? What does that imply for the evaluation
   budget of the ACT vs Diffusion Policy comparison?
   A: Depends on the Wilson CI. When number of trial (episode) is large enough, such that the overlapping area between the 2 CI is small enough (or no overlap at all).

**After discussion (2026-11-19).**
- Q1: picking the best of 10 noisy scores biases the chosen score upward (winner's curse).
  If seed 1 were used for selection, 49/50 would be a validation score. An unbiased number
  needs a fresh seed set evaluated only on the chosen checkpoint, like a test set.
- Q2: CI overlap is the wrong criterion. It is too conservative and ignores pairing:
  140 vs 160 overlap, yet McNemar p = 0.031. Simulated power (α = 0.05): 100% vs 98% needs
  about 500 episodes; 98% vs 90% needs about 100 (failures nested) to 200 (failures
  independent). For ACT vs DP: avoid near-ceiling tasks and budget about 200 paired
  episodes per condition, fixed in advance.
